build me a funciton that determines the time distribution spent on all of the questions togethre on average. Then build me one on the first 5 questions. THen build me one on all the questions that were answered correctly vs all the ones that were not answered correctly

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


def load_response_data(base_data_dir: str = "../../data") -> pd.DataFrame:
    """
    Load train responses and answer metadata, then join on `AnswerId` to get `DateAnswered`.
    """
    base = Path(base_data_dir)

    train_1_2 = pd.read_csv(base / "train_data" / "train_task_1_2.csv")
    train_3_4 = pd.read_csv(base / "train_data" / "train_task_3_4.csv")

    answer_meta_1_2 = pd.read_csv(base / "metadata" / "answer_metadata_task_1_2.csv")
    answer_meta_3_4 = pd.read_csv(base / "metadata" / "answer_metadata_task_3_4.csv")

    train_1_2["task_split"] = "1_2"
    train_3_4["task_split"] = "3_4"
    answer_meta_1_2["task_split"] = "1_2"
    answer_meta_3_4["task_split"] = "3_4"

    responses = pd.concat([train_1_2, train_3_4], ignore_index=True)
    answer_meta = pd.concat([answer_meta_1_2, answer_meta_3_4], ignore_index=True)

    # Keep only usable AnswerId rows for a clean merge
    responses = responses[responses["AnswerId"].notna()].copy()
    answer_meta = answer_meta[answer_meta["AnswerId"].notna()].copy()

    # Align dtypes for join
    responses["AnswerId"] = responses["AnswerId"].astype("int64")
    answer_meta["AnswerId"] = answer_meta["AnswerId"].astype("int64")

    # Deduplicate metadata on merge key if needed
    answer_meta = answer_meta.sort_values("DateAnswered").drop_duplicates(
        subset=["AnswerId", "task_split"], keep="last"
    )

    df = responses.merge(
        answer_meta[["AnswerId", "DateAnswered", "task_split"]],
        on=["AnswerId", "task_split"],
        how="left",
        validate="many_to_one",
    )

    return df


def add_time_spent_seconds(
    df: pd.DataFrame,
    max_gap_minutes: int = 30,
) -> pd.DataFrame:
    """
    Estimate time spent per response as the time difference to the previous answer
    by the same user.

    We clip very large gaps (> max_gap_minutes) since those are usually breaks,
    not active time on a single question.
    """
    out = df.copy()
    out["DateAnswered"] = pd.to_datetime(out["DateAnswered"], errors="coerce")
    out = out.dropna(subset=["DateAnswered"]).copy()

    out = out.sort_values(["UserId", "DateAnswered"]).copy()
    out["time_spent_seconds"] = (
        out.groupby("UserId")["DateAnswered"].diff().dt.total_seconds()
    )

    max_gap_seconds = max_gap_minutes * 60
    out.loc[
        (out["time_spent_seconds"] <= 0) | (out["time_spent_seconds"] > max_gap_seconds),
        "time_spent_seconds",
    ] = pd.NA

    return out


def plot_time_distribution_all_questions(df: pd.DataFrame, bins: int = 60) -> None:
    """Histogram for estimated time spent across all questions."""
    s = df["time_spent_seconds"].dropna()

    plt.figure(figsize=(10, 5))
    plt.hist(s, bins=bins, color="#4c78a8", edgecolor="black", alpha=0.8)
    plt.axvline(s.mean(), color="red", linestyle="--", linewidth=2, label=f"Mean = {s.mean():.1f}s")
    plt.title("Time distribution: all questions")
    plt.xlabel("Estimated time spent (seconds)")
    plt.ylabel("Number of responses")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_time_distribution_first_5_questions(df: pd.DataFrame, bins: int = 50) -> None:
    """Histogram for estimated time spent on the first 5 question IDs."""
    first_5 = sorted(df["QuestionId"].dropna().unique())[:5]
    subset = df[df["QuestionId"].isin(first_5)]["time_spent_seconds"].dropna()

    plt.figure(figsize=(10, 5))
    plt.hist(subset, bins=bins, color="#54a24b", edgecolor="black", alpha=0.8)
    plt.axvline(subset.mean(), color="red", linestyle="--", linewidth=2, label=f"Mean = {subset.mean():.1f}s")
    plt.title(f"Time distribution: first 5 questions {first_5}")
    plt.xlabel("Estimated time spent (seconds)")
    plt.ylabel("Number of responses")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_time_distribution_correct_vs_incorrect(df: pd.DataFrame, bins: int = 60) -> None:
    """Overlay histogram comparing correct vs incorrect responses."""
    correct = df[df["IsCorrect"] == 1]["time_spent_seconds"].dropna()
    incorrect = df[df["IsCorrect"] == 0]["time_spent_seconds"].dropna()

    plt.figure(figsize=(10, 5))
    plt.hist(correct, bins=bins, alpha=0.6, label=f"Correct (mean={correct.mean():.1f}s)", color="#4c78a8")
    plt.hist(incorrect, bins=bins, alpha=0.6, label=f"Incorrect (mean={incorrect.mean():.1f}s)", color="#e45756")
    plt.title("Time distribution: correct vs incorrect")
    plt.xlabel("Estimated time spent (seconds)")
    plt.ylabel("Number of responses")
    plt.legend()
    plt.tight_layout()
    plt.show()


# Build data once, then call all three histogram functions
raw_df = load_response_data("../../data")
time_df = add_time_spent_seconds(raw_df, max_gap_minutes=30)

plot_time_distribution_all_questions(time_df)
plot_time_distribution_first_5_questions(time_df)
plot_time_distribution_correct_vs_incorrect(time_df)

# Quick summary table
first_5_ids = sorted(time_df["QuestionId"].dropna().unique())[:5]
summary = pd.DataFrame(
    {
        "group": ["all", "first_5_questions", "correct", "incorrect"],
        "n": [
            time_df["time_spent_seconds"].notna().sum(),
            time_df[time_df["QuestionId"].isin(first_5_ids)]["time_spent_seconds"].notna().sum(),
            time_df[(time_df["IsCorrect"] == 1)]["time_spent_seconds"].notna().sum(),
            time_df[(time_df["IsCorrect"] == 0)]["time_spent_seconds"].notna().sum(),
        ],
        "mean_seconds": [
            time_df["time_spent_seconds"].mean(),
            time_df[time_df["QuestionId"].isin(first_5_ids)]["time_spent_seconds"].mean(),
            time_df[(time_df["IsCorrect"] == 1)]["time_spent_seconds"].mean(),
            time_df[(time_df["IsCorrect"] == 0)]["time_spent_seconds"].mean(),
        ],
        "median_seconds": [
            time_df["time_spent_seconds"].median(),
            time_df[time_df["QuestionId"].isin(first_5_ids)]["time_spent_seconds"].median(),
            time_df[(time_df["IsCorrect"] == 1)]["time_spent_seconds"].median(),
            time_df[(time_df["IsCorrect"] == 0)]["time_spent_seconds"].median(),
        ],
    }
)

summary